In [1]:
import os
os.environ['LOGS_DIRECTORY'] = '../eval_logs'

import sys
sys.path.insert(0, '..')

In [2]:
import json
import random

from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent

from main import initialize_index, initialize_agent
from logs import log_interaction_to_file, LOG_DIR

In [3]:
LOG_DIR.absolute()

PosixPath('/mnt/c/sha01_learn/aihero/app/eval/../eval_logs')

In [4]:
index = initialize_index()
agent = initialize_agent(index)

Starting AI FAQ Assistant for dbt-labs/docs.getdbt.com
Initializing data ingestion...
Data indexing completed successfully!
Initializing search agent...
Agent initialized successfully!


In [5]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='gpt-4o-mini',
    output_type=QuestionsList
)

In [6]:
sample = random.sample(index.docs, 10)
prompt = [d['content'] for d in sample]

In [7]:
result = await question_generator.run(user_prompt=json.dumps(prompt))
question_list = result.output

In [8]:
for question in tqdm(question_list.questions):
    print(question)
    result = await agent.run(user_prompt=question)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages(), source='ai-generated')
    print()

  0%|          | 0/15 [00:00<?, ?it/s]

What are the main advantages of using views in dbt projects?
Using views in dbt projects has several main advantages:

1. **Simplicity and Readability**: Views allow you to create a simple representation of your data without altering the underlying tables. This can help keep your SQL queries straightforward and easier to read since views can encapsulate complex SQL logic.

2. **Real-time Data Access**: When you use views, any changes made to the underlying source data are immediately available to users querying the view. This ensures that the most current data is being accessed at all times without the need to refresh or reload any tables.

3. **No Additional Storage Costs**: Unlike tables, views do not store data independently. This can be beneficial for conserving storage space, particularly when working with large datasets.

4. **Optimal for Less Complex Transformations**: Views are typically best for queries that do not require significant transformations. For example, if your mode